In [1]:
import os
import pandas as pd
import numpy as np
from skmultilearn.model_selection import iterative_train_test_split

# 1. CẤU HÌNH DATASET
DATASET_NAME = 'VinDr-PCXR' # VinDr-CXR, VinDr-PCXR

RAW_DIR = f"../data/raw/{DATASET_NAME}/annotations"
PROCESSED_DIR = f"../data/processed/{DATASET_NAME}/labels"

os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"🚀 BẮT ĐẦU XỬ LÝ BỘ: {DATASET_NAME}")

🚀 BẮT ĐẦU XỬ LÝ BỘ: VinDr-PCXR


In [2]:
print("--- PHẦN 1: TẠO TẬP TRAIN & VAL TÁCH BIỆT ---")

# 1. Đọc file Train gốc
train_raw_path = os.path.join(RAW_DIR, "image_labels_train.csv")
df_train_raw = pd.read_csv(train_raw_path)

# 2. XÁC ĐỊNH DANH SÁCH NHÃN CHUẨN (BỎ NO FINDING)
if DATASET_NAME == 'VinDr-CXR':
    # Chỉ lấy đúng 14 nhãn chuẩn Kaggle 
    target_classes = [
        'Aortic enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly',
        'Consolidation', 'ILD', 'Infiltration', 'Lung Opacity', 'Nodule/Mass',
        'Other lesion', 'Pleural effusion', 'Pleural thickening', 'Pneumothorax',
        'Pulmonary fibrosis'
    ]
elif DATASET_NAME == 'VinDr-PCXR':
    # Trẻ em: Bỏ 'rad_ID' và 'No finding'
    exclude_cols = ['image_id', 'rad_id', 'rad_ID', 'study_id', 'patient_id', 'No finding', 'No findings']
    target_classes = [col for col in df_train_raw.columns if col not in exclude_cols]

print(f"[*] Tổng số bệnh lý đưa vào huấn luyện: {len(target_classes)} classes")
print(f"[*] Danh sách: {target_classes}")

--- PHẦN 1: TẠO TẬP TRAIN & VAL TÁCH BIỆT ---
[*] Tổng số bệnh lý đưa vào huấn luyện: 14 classes
[*] Danh sách: ['Bronchitis', 'Brocho-pneumonia', 'Other disease', 'Bronchiolitis', 'Situs inversus', 'Pneumonia', 'Pleuro-pneumonia', 'Diagphramatic hernia', 'Tuberculosis', 'Congenital emphysema', 'CPAM', 'Hyaline membrane disease', 'Mediastinal tumor', 'Lung tumor']


In [3]:
# 3. MAJORITY VOTE (Gộp bác sĩ)

# Gom nhóm theo image_id và tính tổng số vote của các bác sĩ
df_votes = df_train_raw.groupby('image_id', as_index=True)[target_classes].sum()

# Tự động xác định ngưỡng đồng thuận (Threshold)
# VinDr-CXR (người lớn): 3 bác sĩ -> Lấy Majority Vote (>= 2)
# VinDr-PCXR (trẻ em): 1 bác sĩ -> Lấy nhãn gốc (>= 1)

if 'DATASET_NAME' in locals() and DATASET_NAME == 'VinDr-CXR':
    vote_threshold = 2
    print("[*] VinDr-CXR: Áp dụng Majority Vote (>= 2 bác sĩ đồng thuận).")
elif 'DATASET_NAME' in locals() and DATASET_NAME == 'VinDr-PCXR':
    vote_threshold = 1
    print("[*] VinDr-PCXR: 1 bác sĩ gán nhãn -> Giữ nguyên nhãn gốc (>= 1).")
else:
    # Tự động nhận diện dựa trên số lượt gán nhãn thực tế trên mỗi ảnh
    max_annotators = df_train_raw.groupby('image_id').size().max()
    vote_threshold = 2 if max_annotators >= 3 else 1
    print(f"[*] Tự động nhận diện: Số bác sĩ tối đa/ảnh = {max_annotators} -> Ngưỡng vote = {vote_threshold}")

# Áp dụng ngưỡng đã chọn
df_train_clean = (df_votes >= vote_threshold).astype(int).reset_index()

print(f"[*] Tổng số ảnh Train gốc sau gộp: {len(df_train_clean)}")

[*] VinDr-PCXR: 1 bác sĩ gán nhãn -> Giữ nguyên nhãn gốc (>= 1).
[*] Tổng số ảnh Train gốc sau gộp: 7728


In [4]:
# 4. TÁCH HOÀN TOÀN 10% VAL TỪ TRAIN
X = df_train_clean[['image_id']].values
y = df_train_clean[target_classes].values

X_train, y_train, X_val, y_val = iterative_train_test_split(X, y, test_size=0.1)

df_train_split = pd.DataFrame(X_train, columns=['image_id'])
df_train_split[target_classes] = y_train

df_val_split = pd.DataFrame(X_val, columns=['image_id'])
df_val_split[target_classes] = y_val

In [5]:
# 5. SANITY CHECK (CHỐNG RÒ RỈ DỮ LIỆU)
train_ids = set(df_train_split['image_id'])
val_ids = set(df_val_split['image_id'])
overlap_ids = train_ids.intersection(val_ids)

print(f"\n-> Kích thước Train 90%: {len(train_ids)} ảnh")
print(f"-> Kích thước Val 10%:   {len(val_ids)} ảnh")
print(f"-> Số ảnh bị trùng lặp:   {len(overlap_ids)}")

if len(overlap_ids) == 0:
    print("✅ TẬP VAL ĐÃ TÁCH BIỆT HOÀN TOÀN (NO DATA LEAKAGE).")
else:
    raise ValueError("LỖI: Phát hiện rò rỉ dữ liệu!")



-> Kích thước Train 90%: 6955 ảnh
-> Kích thước Val 10%:   773 ảnh
-> Số ảnh bị trùng lặp:   0
✅ TẬP VAL ĐÃ TÁCH BIỆT HOÀN TOÀN (NO DATA LEAKAGE).


In [6]:
# 6. XỬ LÝ TẬP TEST GỐC
print("\n--- PHẦN 2: XỬ LÝ TẬP TEST ---")
test_raw_path = os.path.join(RAW_DIR, "image_labels_test.csv")

if os.path.exists(test_raw_path):
    df_test_raw = pd.read_csv(test_raw_path)
    # Cắt lấy đúng các cột nhãn giống tập Train
    df_test_clean = df_test_raw[['image_id'] + target_classes].copy()
    df_test_clean = df_test_clean.drop_duplicates(subset=['image_id']).reset_index(drop=True)
    
    # Lưu Test
    test_out = os.path.join(PROCESSED_DIR, "test_labels.csv")
    df_test_clean.to_csv(test_out, index=False)
    print(f"✅ Đã xử lý xong tập Test ({len(df_test_clean)} ảnh) -> {test_out}")
else:
    print(f"⚠️ Không tìm thấy file Test gốc tại {test_raw_path}")

# LƯU FILE TRAIN & VAL
train_out = os.path.join(PROCESSED_DIR, "train_split.csv")
val_out = os.path.join(PROCESSED_DIR, "val_split.csv")
df_train_split.to_csv(train_out, index=False)
df_val_split.to_csv(val_out, index=False)

print("\n🚀 ĐÃ HOÀN TẤT!")
print(f"Bạn có thể xem dữ liệu đã xử lý tại: {PROCESSED_DIR}")


--- PHẦN 2: XỬ LÝ TẬP TEST ---
✅ Đã xử lý xong tập Test (1397 ảnh) -> ../data/processed/VinDr-PCXR/labels\test_labels.csv

🚀 ĐÃ HOÀN TẤT!
Bạn có thể xem dữ liệu đã xử lý tại: ../data/processed/VinDr-PCXR/labels
